# NeuroScan Nepal — GPU Training (Colab)

**Before running:** Runtime → Change runtime type → **GPU (T4)**

Trains the same `BaselineCNN` as your local project. Download outputs and copy to `models/` on your PC.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Enable GPU: Runtime → Change runtime type → GPU')

In [ ]:
!pip install -q opencv-python-headless scikit-learn matplotlib

## Configuration

Upload **`neuroscan_data.zip`** to Google Drive with structure:
```
raw/normal/...
raw/abnormal/...
```

Optional: put `cnn_baseline.pth` in the same Drive folder for fine-tuning.

In [ ]:
from pathlib import Path
from google.colab import drive, files
import zipfile

WORK = Path('/content/neuroscan')
WORK.mkdir(exist_ok=True)

# --- Option A: zip already on Google Drive (edit folder name if needed) ---
USE_DRIVE = True
DRIVE_FOLDER = Path('/content/drive/MyDrive/NeuroScan')
DRIVE_ZIP = DRIVE_FOLDER / 'neuroscan_data.zip'
PRETRAINED = DRIVE_FOLDER / 'cnn_baseline.pth'  # optional

DATA_ZIP = WORK / 'neuroscan_data.zip'

if USE_DRIVE:
    drive.mount('/content/drive')
    if DRIVE_ZIP.exists():
        DATA_ZIP = DRIVE_ZIP
        print('Using zip from Drive:', DATA_ZIP)
    else:
        print('Not found on Drive:', DRIVE_ZIP)
        print('Upload neuroscan_data.zip from your PC now...')
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError(
                'Create zip on PC: run scripts/zip_dataset_for_colab.bat, '
                'then upload here or to My Drive/NeuroScan/'
            )
        name = next(iter(uploaded))
        DATA_ZIP.write_bytes(uploaded[name])
        print('Saved upload to:', DATA_ZIP)
else:
    print('Upload neuroscan_data.zip from your PC...')
    uploaded = files.upload()
    name = next(iter(uploaded))
    DATA_ZIP.write_bytes(uploaded[name])

LOAD_PRETRAINED = PRETRAINED.exists()
FINE_TUNE = LOAD_PRETRAINED

with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
    zf.extractall(WORK)

if (WORK / 'raw' / 'normal').exists():
    raw = WORK / 'raw'
elif (WORK / 'data' / 'raw' / 'normal').exists():
    raw = WORK / 'data' / 'raw'
elif (WORK / 'normal').exists():
    raw = WORK
else:
    raise FileNotFoundError('Zip must contain normal/ and abnormal/ folders')

normal_count = len(list((raw / 'normal').rglob('*.*')))
abnormal_count = len(list((raw / 'abnormal').rglob('*.*')))
print(f'Normal: {normal_count}, Abnormal: {abnormal_count}')
print('Pretrained exists:', PRETRAINED.exists())

In [ ]:
# Data was extracted in the cell above — nothing to run here.
# If you see FileNotFoundError, re-run the previous cell and upload neuroscan_data.zip when prompted.
print('Data ready at:', raw)

## Upload training code

Run the next cell, then use the file picker to upload:
- `cnn_baseline.py`
- `dataset_utils.py`
- `preprocessing.py`

From your PC: `NeuroScan_Nepal/src/`

In [ ]:
from google.colab import files
import shutil

src_dir = WORK / 'src'
src_dir.mkdir(exist_ok=True)

uploaded = files.upload()  # select the 3 .py files
for name, data in uploaded.items():
    (src_dir / name).write_bytes(data)
    print('Saved', name)

required = ['cnn_baseline.py', 'dataset_utils.py', 'preprocessing.py']
missing = [f for f in required if not (src_dir / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing: {missing}')

In [ ]:
import sys
sys.path.insert(0, str(WORK / 'src'))

from cnn_baseline import train_baseline_cnn, BaselineCNN
import torch

models_dir = WORK / 'models'
results_dir = WORK / 'results'
models_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

model_path = models_dir / 'cnn_baseline.pth'
if PRETRAINED.exists():
    shutil.copy2(PRETRAINED, model_path)
    print('Copied pretrained weights for fine-tuning')

epochs = 15 if FINE_TUNE else 30
lr = 0.0001 if FINE_TUNE else 0.0005
batch = 64

print(f'Training: epochs={epochs}, lr={lr}, batch={batch}, fine_tune={FINE_TUNE}')

train_baseline_cnn(
    normal_dir=raw / 'normal',
    abnormal_dir=raw / 'abnormal',
    model_path=model_path,
    results_path=results_dir / 'cnn_baseline_report.txt',
    epochs=epochs,
    batch_size=batch,
    learning_rate=lr,
    weight_decay=0.0001,
    label_smoothing=0.03,
    early_stopping_patience=5 if FINE_TUNE else 7,
    augment=True,
    cache=True,
    use_clahe=True,
    random_state=42,
)

print('\n--- Report ---')
print((results_dir / 'cnn_baseline_report.txt').read_text())

In [ ]:
# Copy results to Drive + download to PC
out_drive = DRIVE_FOLDER / 'colab_outputs'
out_drive.mkdir(exist_ok=True)

for name in ['cnn_baseline.pth', 'cnn_baseline_calibration.json', 'cnn_baseline_previous.pth']:
    src = models_dir / name
    if src.exists():
        shutil.copy2(src, out_drive / name)
        print('Drive:', out_drive / name)

for name in ['cnn_baseline_report.txt', 'cnn_baseline_split.json', 'training_curves.png']:
    src = results_dir / name
    if src.exists():
        shutil.copy2(src, out_drive / name)

from google.colab import files
files.download(str(models_dir / 'cnn_baseline.pth'))
cal = models_dir / 'cnn_baseline_calibration.json'
if cal.exists():
    files.download(str(cal))
files.download(str(results_dir / 'cnn_baseline_report.txt'))

print('Done! Copy downloaded files to NeuroScan_Nepal/models/ and results/ on your PC.')